# FastSpeech2VN on Kaggle
Notebook nay setup, preprocess, train va infer WAV cho FastSpeech2 tieng Viet (InfoRe1).

Recommended accelerator: **2 x T4** cho repo da duoc toi uu DataParallel + AMP.

In [ ]:
# 1) Clone repo
!git clone https://github.com/A40405/FastSpeech2VN.git
%cd /kaggle/working/FastSpeech2VN

In [ ]:
# 2) Kiem tra GPU
!nvidia-smi

In [ ]:
# 3) Cai dependencies
!python -m pip install --upgrade pip
!pip install -r requirements-kaggle.txt

In [ ]:
# 4) Download dataset InfoRe1 ve thu muc project
!python scripts/download_infore1_dataset.py --dataset doof-ferb/infore1_25hours --split train --output-dir ./corpus/infore1_25hours

In [ ]:
# 5) Prepare alignment input
!python prepare_align.py ./config/InfoRe1_25hours/preprocess.yaml

In [ ]:
# 6) Bootstrap TextGrid
!python scripts/bootstrap_textgrids.py --raw-root ./raw_data/InfoRe1 --output-root ./preprocessed_data/InfoRe1/TextGrid

In [ ]:
# 7) Preprocess acoustic features
!python preprocess.py ./config/InfoRe1_25hours/preprocess.yaml

In [ ]:
# 8) Train FastSpeech2
!python train.py -p ./config/InfoRe1_25hours/preprocess.yaml -m ./config/InfoRe1_25hours/model.yaml -t ./config/InfoRe1_25hours/train.yaml

In [ ]:
# 9) Download HiFi-GAN pretrained weights cho WAV chat luong cao
!python scripts/download_hifigan_pretrained.py

In [ ]:
# 10) Tu lay checkpoint moi nhat va infer
import glob
import re

ckpts = glob.glob('./output/ckpt/InfoRe1/*.pth.tar')
assert ckpts, 'Chua co checkpoint. Hay train truoc.'
step = max(int(re.search(r'(\d+)\.pth\.tar$', c).group(1)) for c in ckpts)
print('Using checkpoint step:', step)

!python synthesize.py --mode single --text "xin chao, day la fastspeech hai tieng viet" --restore_step {step} -p ./config/InfoRe1_25hours/preprocess.yaml -m ./config/InfoRe1_25hours/model.yaml -t ./config/InfoRe1_25hours/train.yaml